# 00 · bootstrap on the compute machine
Pull the latest code, check the environment, reuse an existing warm cache, run the quick selftest.
Run every cell top to bottom; nothing here is expensive.

In [ ]:
# latest code from GitHub (the Mac pushes, this machine pulls)
!git -C "$(git rev-parse --show-toplevel)" pull --ff-only
!git -C "$(git rev-parse --show-toplevel)" log --oneline -3

In [ ]:
import os, sys, json, time
from pathlib import Path
HERE = Path.cwd()                       # this notebook lives in experiments/simple
assert (HERE / "runInflow.py").exists(), "run this notebook from experiments/simple (Jupyter's cwd is the notebook's folder)"
sys.path.insert(0, str(HERE))
ROOT = HERE / "sweep_results"           # results root (committed to git; warm_cache/ and smoke/ are ignored)
import runInflow as ri
from sweep import config as C, grids, launcher, analysis
print("cwd:", HERE, "| results:", ROOT)

In [ ]:
# environment: cores, memory, load, python, packages, ffmpeg
import platform, shutil, subprocess
print("host:", platform.node(), "| python:", platform.python_version(), "| cpu_count:", os.cpu_count())
print("load (1/5/15 min):", os.getloadavg())
print("free memory (GB):", round(launcher.mem_available_gb(), 1))
for m in ("numpy", "scipy", "matplotlib", "pandas", "numba"):
    try:
        mod = __import__(m); print(f"{m:10s} {mod.__version__}")
    except Exception as e:
        print(f"{m:10s} MISSING ({e})")
print("ffmpeg:", shutil.which("ffmpeg"))
print("who is logged in:"); print(subprocess.run(["who"], capture_output=True, text=True).stdout)

In [ ]:
# reuse an existing warm cache from an older copy of the repo, if there is one under ~ (cache files are keyed by physics + seed)
target = HERE / "warm_cache"
if not target.exists():
    found = [p for p in Path.home().rglob("warm_cache") if p.is_dir() and p != target and any(p.glob("warm_s*.npz"))]
    if found:
        target.symlink_to(found[0], target_is_directory=True); print("linked", target, "->", found[0])
    else:
        target.mkdir(); print("no existing warm cache found; created", target)
print("warm cache files:", len(list(target.glob("warm_s*.npz"))))

In [ ]:
ri.selftest_geometry(); print("geometry selftest passed")
!python -m sweep.selftest